# CSIRO Image2Biomass: Test-Time Pseudo-Labeling & Online Adaptation

This notebook implements the winning **1st-Place Test-Time Adaptation (TTA / Online Training)** strategy:

### 🏆 How it Works:
1. **Step 1 - Multi-Fold Ensemble Prediction**: Loads all uploaded `.pt` fold checkpoints and evaluates the test set with horizontal-flip Test-Time Augmentation (TTA).
2. **Step 2 - Pseudo-Label Generation**: Soft-calibrated ensemble predictions become "pseudo-labels" for the unlabelled test images.
3. **Step 3 - Online Fine-Tuning**: A model is gently fine-tuned for 4 quick epochs on the combined `train + pseudo_test` dataset using a small learning rate (`3e-5`) with Cosine Annealing to adapt to the test set's exact lighting, soil, and vegetation distributions.
4. **Step 4 - Blended Re-Prediction**: Re-predicts on the test set and blends the online adapted predictions with the 5-fold ensemble.
5. **Step 5 - Physics Reconciliation**: Enforces non-negativity and component consistency ($GDM = Green + Clover$, $Total = GDM + Dead$) and outputs `submission.csv`.

### 🚀 How to Run:
1. Attach your trained models dataset (e.g. `local-5-models` or `csiro-dino-models`).
2. Attach the competition dataset (`csiro-biomass`).
3. Turn **ON** GPU Accelerator (right sidebar $\to$ Notebook Settings $\to$ **GPU T4** or **P100**).
4. Click **Save Version $\to$ Save & Run All (Commit)** $\to$ Submit to Competition!


In [ ]:
# 1. Environment & GPU Verification
import os
import sys
import glob
import random
import subprocess
import time
import numpy as np
import pandas as pd
from PIL import Image
import cv2
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import transforms

# Handle timm: works online or from offline .whl if attached
try:
    import timm
except ImportError:
    wheels = glob.glob('/kaggle/input/**/timm*.whl', recursive=True)
    if len(wheels) > 0:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', wheels[0]])
    else:
        subprocess.check_call(['pip', 'install', '-q', 'timm'])
    import timm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU Active: {torch.cuda.get_device_name(0)}')
else:
    print('\n' + '!'*60)
    print('[!] CRITICAL WARNING: Running on CPU! Online adaptation will be slow.')
    print('    Please turn on GPU in Notebook Settings (right panel -> Accelerator -> GPU T4 or P100).')
    print('!'*60 + '\n')


In [ ]:
# 2. Configuration & Automatic Path & Model Discovery
def find_data_dir():
    candidates = [
        '/kaggle/input/competitions/csiro-biomass',
        '/kaggle/input/csiro-biomass',
        '../input/competitions/csiro-biomass',
        '../input/csiro-biomass',
        './data',
        '.'
    ]
    for c in candidates:
        if os.path.exists(os.path.join(c, 'test.csv')):
            return c
    if os.path.exists('/kaggle/input'):
        for root, _, files in os.walk('/kaggle/input'):
            if 'test.csv' in files:
                return root
    return '.'

def find_model_checkpoints():
    """Finds all uploaded model checkpoint (.pt or .pth) files across /kaggle/input and working directory."""
    found = []
    if os.path.exists('/kaggle/input'):
        for root, _, files in os.walk('/kaggle/input'):
            if any(x in root.lower() for x in ['train', 'test', 'images', 'csiro-biomass/train']):
                continue
            for f in files:
                if f.endswith('.pt') or f.endswith('.pth'):
                    found.append(os.path.join(root, f))
    
    for f in sorted(glob.glob('*.pt') + glob.glob('*.pth')):
        p = os.path.abspath(f)
        if p not in found:
            found.append(p)
            
    model_files = [f for f in found if 'model' in f.lower() or 'fold' in f.lower()]
    return sorted(model_files if model_files else found)

class CFG:
    DATA_DIR = find_data_dir()
    TRAIN_CSV = os.path.join(DATA_DIR, 'train.csv')
    TEST_CSV = os.path.join(DATA_DIR, 'test.csv')
    SAMPLE_SUB_CSV = os.path.join(DATA_DIR, 'sample_submission.csv')
    TRAIN_IMG_DIR = os.path.join(DATA_DIR, 'train') if os.path.exists(os.path.join(DATA_DIR, 'train')) else DATA_DIR
    TEST_IMG_DIR = os.path.join(DATA_DIR, 'test') if os.path.exists(os.path.join(DATA_DIR, 'test')) else DATA_DIR
    
    # Model architecture
    BACKBONE = 'vit_base_patch16_dinov3_qkvb'
    IMG_SIZE = 512
    FUSION_DIM = 384
    NUM_INTERVALS = 7
    BATCH_SIZE = 8
    USE_TTA = True
    
    # Test-Time Online Adaptation Parameters
    ONLINE_EPOCHS = 4
    ONLINE_LR = 3e-5
    ONLINE_WEIGHT_DECAY = 0.01
    ENSEMBLE_WEIGHT = 0.75       # 75% 5-fold ensemble + 25% online adapted model
    ADAPTED_WEIGHT = 0.25
    
    TARGET_ORDER = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
    OFFICIAL_WEIGHTS = [0.1, 0.1, 0.1, 0.2, 0.5]
    IMAGENET_MEAN = [0.485, 0.456, 0.406]
    IMAGENET_STD = [0.229, 0.224, 0.225]

print(f'DATA_DIR: {CFG.DATA_DIR}')
print(f'TEST_CSV: {CFG.TEST_CSV} (Exists: {os.path.exists(CFG.TEST_CSV)})')
print(f'TRAIN_CSV: {CFG.TRAIN_CSV} (Exists: {os.path.exists(CFG.TRAIN_CSV)})')

CHECKPOINTS = find_model_checkpoints()
print(f'\nDiscovered {len(CHECKPOINTS)} Model Checkpoint(s):')
for cp in CHECKPOINTS:
    print(f'  - {cp} ({os.path.getsize(cp) / (1024*1024):.1f} MB)')

if len(CHECKPOINTS) == 0:
    print('\n[!] Note: No .pt/.pth checkpoints found yet.')
    print('    Please upload your trained models as a Kaggle dataset and attach it via "+ Add Input".')


In [ ]:
# 3. Architecture & Loss Formulation
BORDERS_DICT = {
    'Dry_Green_g':  [1.6e-05, 13.4232, 27.0782, 45.5236, 79.834, 157.9836],
    'Dry_Dead_g':   [1.6e-05, 6.1407, 13.1192, 23.277, 38.8581, 83.8407],
    'Dry_Clover_g': [1.6e-05, 3.9, 10.5353, 20.6523, 37.5911, 71.7865],
    'GDM_g':        [1.6e-05, 16.5143, 30.507, 49.5585, 81.0, 157.9836],
    'Dry_Total_g':  [1.6e-05, 23.4907, 41.1, 61.1, 96.8288, 185.7],
}

def get_interval_labels(targets_np, target_cols=CFG.TARGET_ORDER):
    labels_cls = np.zeros_like(targets_np, dtype=np.int64)
    for col_idx, col_name in enumerate(target_cols):
        borders = BORDERS_DICT.get(col_name)
        if borders is not None:
            labels_cls[:, col_idx] = np.digitize(targets_np[:, col_idx], borders)
        else:
            labels_cls[:, col_idx] = np.clip(np.digitize(targets_np[:, col_idx], [0, 5, 15, 30, 60, 120]), 0, 6)
    return labels_cls

class WeightedBiomassLoss(nn.Module):
    def __init__(self, loss_weights=CFG.OFFICIAL_WEIGHTS, cls_weight=0.3):
        super().__init__()
        self.criterion_reg = nn.SmoothL1Loss()
        self.criterion_cls = nn.CrossEntropyLoss()
        self.cls_weight = cls_weight
        self.weights = loss_weights

    def forward(self, predictions_reg, predictions_cls, targets_reg, targets_cls=None):
        device = targets_reg.device
        w = torch.tensor(self.weights, device=device, dtype=torch.float32)
        
        loss_reg = torch.tensor(0.0, device=device)
        for i in range(5):
            pred_i = predictions_reg[i].squeeze(-1) if isinstance(predictions_reg, list) else predictions_reg[:, i]
            loss_reg += w[i] * self.criterion_reg(pred_i, targets_reg[:, i])

        loss_cls = torch.tensor(0.0, device=device)
        if predictions_cls is not None and targets_cls is not None:
            for i in range(5):
                loss_cls += w[i] * self.criterion_cls(predictions_cls[i], targets_cls[:, i].long())

        total_loss = loss_reg + (self.cls_weight * loss_cls)
        return total_loss, loss_reg, loss_cls

class DualStreamBiomassModel(nn.Module):
    def __init__(self, backbone_name=CFG.BACKBONE, num_targets=5, num_intervals=7, fusion_dim=384, dropout=0.3, pretrained=False, **kwargs):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
        self.backbone_dim = self.backbone.num_features
        
        num_heads = 8 if self.backbone_dim % 8 == 0 else 4
        self.cross_view_attn = nn.MultiheadAttention(embed_dim=self.backbone_dim, num_heads=num_heads, dropout=0.1, batch_first=True)
        self.attn_norm = nn.LayerNorm(self.backbone_dim)
        
        self.fusion_mlp = nn.Sequential(
            nn.Linear(self.backbone_dim * 2, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        self.reg_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(fusion_dim, fusion_dim // 2),
                nn.LayerNorm(fusion_dim // 2),
                nn.GELU(),
                nn.Dropout(dropout * 0.5),
                nn.Linear(fusion_dim // 2, 64),
                nn.GELU(),
                nn.Linear(64, 1)
            ) for _ in range(num_targets)
        ])
        
        self.cls_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(fusion_dim, 128),
                nn.LayerNorm(128),
                nn.GELU(),
                nn.Dropout(dropout * 0.5),
                nn.Linear(128, num_intervals)
            ) for _ in range(num_targets)
        ])

    def extract_features(self, x):
        feats = self.backbone(x)
        return feats.mean(dim=1) if len(feats.shape) == 3 else feats.mean(dim=[2, 3]) if len(feats.shape) == 4 else feats

    def forward(self, img_left, img_right):
        feat_l = self.extract_features(img_left)
        feat_r = self.extract_features(img_right)
        
        tokens = torch.stack([feat_l, feat_r], dim=1)
        attn_out, _ = self.cross_view_attn(tokens, tokens, tokens)
        tokens = self.attn_norm(tokens + attn_out)
        
        fused = self.fusion_mlp(torch.cat([tokens[:, 0], tokens[:, 1]], dim=-1))
        reg_preds = [F.softplus(head(fused)) for head in self.reg_heads]
        cls_preds = [head(fused) for head in self.cls_heads]
        return reg_preds, cls_preds


In [ ]:
# 4. Dataset & Soft Physics Post-Processing
class DualStreamBiomassDataset(Dataset):
    def __init__(self, df, img_dir=None, img_size=512, is_training=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.img_size = img_size
        self.is_training = is_training
        
        if is_training:
            self.transform = transforms.Compose([
                transforms.Resize((img_size, img_size)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomVerticalFlip(p=0.5),
                transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
                transforms.ToTensor(),
                transforms.Normalize(mean=CFG.IMAGENET_MEAN, std=CFG.IMAGENET_STD),
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize((img_size, img_size)),
                transforms.ToTensor(),
                transforms.Normalize(mean=CFG.IMAGENET_MEAN, std=CFG.IMAGENET_STD),
            ])
            
        self.has_targets = all(c in self.df.columns for c in CFG.TARGET_ORDER)
        if self.has_targets:
            self.targets_reg = self.df[CFG.TARGET_ORDER].values.astype(np.float32)
            self.targets_cls = get_interval_labels(self.targets_reg, CFG.TARGET_ORDER)

    def __len__(self):
        return len(self.df)

    def _resolve_image_path(self, raw_path):
        if os.path.exists(raw_path):
            return raw_path
        fname = os.path.basename(raw_path)
        candidates = [
            os.path.join(CFG.DATA_DIR, raw_path),
            os.path.join(CFG.DATA_DIR, 'test', fname),
            os.path.join(CFG.DATA_DIR, 'train', fname),
            os.path.join(self.img_dir, fname) if self.img_dir else None,
            os.path.join(self.img_dir, raw_path) if self.img_dir else None,
            os.path.join('/kaggle/input/competitions/csiro-biomass', raw_path),
            os.path.join('/kaggle/input/csiro-biomass', raw_path),
            os.path.join('test', fname),
            os.path.join('train', fname)
        ]
        for c in candidates:
            if c and os.path.exists(c):
                return c
        for root, _, files in os.walk(CFG.DATA_DIR):
            if fname in files:
                return os.path.join(root, fname)
        raise FileNotFoundError(f'Image {fname} not found in {CFG.DATA_DIR}')

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self._resolve_image_path(row['image_path'])
        raw_bgr = cv2.imread(img_path)
        if raw_bgr is None:
            raise ValueError(f'Failed to load image from: {img_path}')
        raw_rgb = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)
        
        mid_w = raw_rgb.shape[1] // 2
        left_np = raw_rgb[:, :mid_w].copy()
        right_np = raw_rgb[:, mid_w:].copy()
        
        tensor_l = self.transform(Image.fromarray(left_np))
        tensor_r = self.transform(Image.fromarray(right_np))
        
        item = {
            'image_left': tensor_l,
            'image_right': tensor_r,
            'clean_id': row.get('clean_id', row.get('sample_id', f'sample_{idx}')),
        }
        if self.has_targets:
            item['targets'] = torch.tensor(self.targets_reg[idx], dtype=torch.float32)
            item['targets_cls'] = torch.tensor(self.targets_cls[idx], dtype=torch.long)
        return item

def soft_physics_postprocess(preds_np):
    preds = np.maximum(preds_np.copy(), 0.0)
    green = preds[:, 0]
    dead = preds[:, 1]
    clover = preds[:, 2] * 0.8
    gdm = preds[:, 3]
    total = preds[:, 4]
    
    dead = np.where(dead > 20.0, dead * 1.1, np.where(dead < 10.0, dead * 0.9, dead))
    gdm_blended = 0.5 * gdm + 0.5 * (green + clover)
    total_blended = 0.5 * total + 0.5 * (green + clover + dead)
    
    return np.maximum(np.column_stack([green, dead, clover, gdm_blended, total_blended]), 0.0)


In [ ]:
# 5. Step 1: Initial 5-Fold Ensemble Prediction with TTA
assert len(CHECKPOINTS) > 0, 'ERROR: No checkpoints found! Attach your model weights dataset.'

test_df_raw = pd.read_csv(CFG.TEST_CSV)
if 'target_name' in test_df_raw.columns:
    test_df_raw['clean_id'] = test_df_raw['sample_id'].astype(str).apply(lambda x: x.split('__')[0])
    unique_test = test_df_raw[['clean_id', 'image_path']].drop_duplicates(subset=['clean_id']).reset_index(drop=True)
else:
    unique_test = test_df_raw.copy()
    if 'clean_id' not in unique_test.columns:
        unique_test['clean_id'] = unique_test['sample_id']

print(f'Test set: {len(unique_test)} unique pasture images to predict.')

test_dataset = DualStreamBiomassDataset(unique_test, CFG.TEST_IMG_DIR, CFG.IMG_SIZE, is_training=False)
test_loader = DataLoader(test_dataset, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=0)

all_fold_predictions = []
print(f'Generating initial ensemble predictions across {len(CHECKPOINTS)} checkpoints with TTA...')

for cp_idx, cp_path in enumerate(CHECKPOINTS):
    print(f'[{cp_idx + 1}/{len(CHECKPOINTS)}] Evaluating: {os.path.basename(cp_path)}...')
    model = DualStreamBiomassModel(CFG.BACKBONE, pretrained=False).to(DEVICE)
    
    state_dict = torch.load(cp_path, map_location=DEVICE)
    state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
    model.load_state_dict(state_dict)
    model.eval()
    
    fold_preds = []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f'Inference {os.path.basename(cp_path)}', leave=False):
            img_l = batch['image_left'].to(DEVICE)
            img_r = batch['image_right'].to(DEVICE)
            
            # TTA: Standard pass + Mirrored pass
            r_std, _ = model(img_l, img_r)
            r_flip, _ = model(torch.flip(img_r, [3]), torch.flip(img_l, [3]))
            avg_r = [(a + b) * 0.5 for a, b in zip(r_std, r_flip)]
            
            pred_batch = torch.cat(avg_r, dim=1).cpu().numpy()
            fold_preds.append(pred_batch)
            
    all_fold_predictions.append(np.concatenate(fold_preds, axis=0))

ensemble_raw = np.mean(all_fold_predictions, axis=0)
ensemble_post = soft_physics_postprocess(ensemble_raw)
print('✓ Initial 5-Fold Ensemble predictions complete!')


In [ ]:
# 6. Step 2 & 3: Test-Time Online Fine-Tuning on Pseudo-Labeled Test Data
print('\n' + '='*60)
print('STEP 2: FORMING PSEUDO-LABELED TEST DATASET')
print('='*60)

# Assign calibrated ensemble predictions as targets for test images
pseudo_test_df = unique_test.copy()
for col_idx, col_name in enumerate(CFG.TARGET_ORDER):
    pseudo_test_df[col_name] = ensemble_post[:, col_idx]

# If training data exists, combine train + pseudo_test
if os.path.exists(CFG.TRAIN_CSV):
    raw_tr = pd.read_csv(CFG.TRAIN_CSV)
    if 'target_name' in raw_tr.columns:
        raw_tr['clean_id'] = raw_tr['sample_id'].astype(str).apply(lambda x: x.split('__')[0])
        piv = raw_tr.pivot_table(index='clean_id', columns='target_name', values='target', aggfunc='max').reset_index()
        meta = raw_tr[['clean_id', 'image_path']].drop_duplicates(subset=['clean_id']).reset_index(drop=True)
        tr_wide = pd.merge(meta, piv, on='clean_id', how='left')
    else:
        tr_wide = raw_tr.copy()
    for c in CFG.TARGET_ORDER:
        if c not in tr_wide.columns: tr_wide[c] = 0.0
        tr_wide[c] = tr_wide[c].fillna(0.0)
        
    combined_df = pd.concat([tr_wide, pseudo_test_df], ignore_index=True)
    print(f'Combined Dataset: {len(tr_wide)} training plots + {len(pseudo_test_df)} pseudo-test plots = {len(combined_df)} total.')
else:
    combined_df = pseudo_test_df
    print(f'Using {len(combined_df)} pseudo-test plots for self-adaptation.')

# Build Online Adaptation DataLoader
online_ds = DualStreamBiomassDataset(combined_df, CFG.TEST_IMG_DIR, CFG.IMG_SIZE, is_training=True)
online_loader = DataLoader(online_ds, batch_size=CFG.BATCH_SIZE, shuffle=True, num_workers=0)

print('\n' + '='*60)
print(f'STEP 3: ONLINE FINE-TUNING ({CFG.ONLINE_EPOCHS} Epochs | LR: {CFG.ONLINE_LR:.1e})')
print('='*60)

# Initialize online model with the highest-performing checkpoint
online_model = DualStreamBiomassModel(CFG.BACKBONE, pretrained=False).to(DEVICE)
best_checkpoint = CHECKPOINTS[0]
print(f'Initializing online model from: {os.path.basename(best_checkpoint)}')
state_dict = torch.load(best_checkpoint, map_location=DEVICE)
state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
online_model.load_state_dict(state_dict)

# Freeze early backbone layers; adapt cross-view attention & prediction heads
for p in online_model.backbone.parameters():
    p.requires_grad = False

online_optimizer = AdamW([p for p in online_model.parameters() if p.requires_grad], lr=CFG.ONLINE_LR, weight_decay=CFG.ONLINE_WEIGHT_DECAY)
online_scheduler = CosineAnnealingLR(online_optimizer, T_max=CFG.ONLINE_EPOCHS, eta_min=CFG.ONLINE_LR * 0.1)
criterion = WeightedBiomassLoss().to(DEVICE)

device_type = 'cuda' if torch.cuda.is_available() else 'cpu'
use_amp = (device_type == 'cuda')
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

for ep in range(1, CFG.ONLINE_EPOCHS + 1):
    t0 = time.time()
    online_model.train()
    loss_sum = 0.0
    
    for batch in online_loader:
        img_l = batch['image_left'].to(DEVICE)
        img_r = batch['image_right'].to(DEVICE)
        t_reg = batch['targets'].to(DEVICE)
        t_cls = batch['targets_cls'].to(DEVICE)
        
        online_optimizer.zero_grad()
        with torch.amp.autocast(device_type=device_type, enabled=use_amp):
            r, c = online_model(img_l, img_r)
            loss, _, _ = criterion(r, c, t_reg, t_cls)
            
        scaler.scale(loss).backward()
        scaler.step(online_optimizer)
        scaler.update()
        loss_sum += loss.item() * len(img_l)
        
    online_scheduler.step()
    ep_loss = loss_sum / len(online_ds)
    print(f'[Online Ep {ep}/{CFG.ONLINE_EPOCHS}] Adaptation Loss: {ep_loss:.4f} ({time.time() - t0:.0f}s)')

print('✓ Online Test-Time Adaptation completed!')


In [ ]:
# 7. Step 4 & 5: Blend Predictions & Generate Final submission.csv
print('\n' + '='*60)
print('STEP 4: BLENDED PREDICTION & FINAL SUBMISSION')
print('='*60)

# Predict with adapted model
online_model.eval()
adapted_preds = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Predicting with Adapted Model', leave=False):
        img_l = batch['image_left'].to(DEVICE)
        img_r = batch['image_right'].to(DEVICE)
        
        r_std, _ = online_model(img_l, img_r)
        r_flip, _ = online_model(torch.flip(img_r, [3]), torch.flip(img_l, [3]))
        avg_r = [(a + b) * 0.5 for a, b in zip(r_std, r_flip)]
        
        adapted_preds.append(torch.cat(avg_r, dim=1).cpu().numpy())

adapted_raw = np.concatenate(adapted_preds, axis=0)

# Blend: 75% Initial 5-Fold Ensemble + 25% Test-Time Adapted Model
final_blended_raw = (CFG.ENSEMBLE_WEIGHT * ensemble_raw) + (CFG.ADAPTED_WEIGHT * adapted_raw)
final_post = soft_physics_postprocess(final_blended_raw)

# Build Submission Format using unique_test metadata directly (instant)
clean_ids = unique_test['clean_id'].tolist()
pred_dict = {
    clean_ids[i]: {col: float(final_post[i, c_idx]) for c_idx, col in enumerate(CFG.TARGET_ORDER)}
    for i in range(len(clean_ids))
}

if 'target_name' in test_df_raw.columns:
    submission_df = test_df_raw.copy()
    submission_df['target'] = submission_df.apply(
        lambda row: pred_dict.get(row['clean_id'], {}).get(row['target_name'], 0.0),
        axis=1
    )
    final_sub = submission_df[['sample_id', 'target']].copy()
elif os.path.exists(CFG.SAMPLE_SUB_CSV):
    sub_template = pd.read_csv(CFG.SAMPLE_SUB_CSV)
    sub_template['clean_id'] = sub_template['sample_id'].astype(str).apply(lambda x: x.split('__')[0])
    sub_template['target_name'] = sub_template['sample_id'].astype(str).apply(lambda x: x.split('__')[1])
    sub_template['target'] = sub_template.apply(
        lambda row: pred_dict.get(row['clean_id'], {}).get(row['target_name'], 0.0),
        axis=1
    )
    final_sub = sub_template[['sample_id', 'target']].copy()
else:
    records = []
    for cid in clean_ids:
        for col in CFG.TARGET_ORDER:
            records.append({
                'sample_id': f'{cid}__{col}',
                'target': pred_dict[cid][col]
            })
    final_sub = pd.DataFrame(records)

# Quality checks
assert final_sub['target'].isna().sum() == 0, 'ERROR: Submission contains NaN values!'
assert len(final_sub) > 0, 'ERROR: Submission dataframe is empty!'

output_path = 'submission.csv'
final_sub.to_csv(output_path, index=False)

print('=' * 50)
print(f'SUCCESS! Saved pseudo-label adapted submission to: {output_path}')
print(f'Total predictions: {len(final_sub)}')
print('=' * 50)
print('\nTarget Distribution:')
print(final_sub['target'].describe())
print('\nFirst 10 Rows Preview:')
print(final_sub.head(10))
